# 🎙️ Estúdio de Voz Oficial do Mascote FEFO (Coqui XTTS-v2)
**Notebook de Alta Fidelidade** — Configurado para clonar e sintetizar a voz oficial do FEFO em **Português do Brasil** nativo com o motor **XTTS-v2**.

---
### ⚡ Como Usar (Apenas 1 Clique):
1. Verifique a GPU no menu: **Ambiente de execução** ➔ **Alterar tipo de ambiente de execução** ➔ Selecione **T4 GPU** ➔ **Salvar**.
2. Clique em **Ambiente de execução** ➔ **Executar tudo** (ou atalho `Ctrl + F9`).
3. Em cerca de 2 minutos, o modelo estará carregado, gerará o primeiro áudio para você ouvir e abrirá um painel visual completo com link compartilhável!

In [ ]:
# 1. Instalação Otimizada do Coqui XTTS-v2 e Dependências
import os, torch
os.environ['COQUI_TOS_AGREED'] = '1'

print('=' * 60)
if torch.cuda.is_available():
    print(f'✓ GPU DETECTADA: {torch.cuda.get_device_name(0)}')
    print(f'✓ VRAM Disponível: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ ATENÇÃO: Nenhuma GPU detectada! Ative em: Ambiente de execução -> Alterar tipo -> T4 GPU')
print('=' * 60)

print('[*] Instalando dependências de sistema (espeak-ng)...')
!sudo apt-get update -qq && sudo apt-get install -y -qq espeak-ng

print('[*] Instalando o Coqui XTTS-v2 e dependências...')
!pip install --upgrade coqui-tts gradio soundfile pydub
print('✓ Instalação concluída com sucesso!')



In [ ]:
# 2. Baixar Repositório do FEFO e Carregar o Modelo XTTS-v2 na GPU
import os, sys, torch
%cd /content

try:
    from TTS.api import TTS
except ModuleNotFoundError:
    print('⚠️ Módulo TTS não encontrado na memória imediata. Forçando recarga do ambiente...')
    import site
    from importlib import reload
    reload(site)
    !pip install coqui-tts
    from TTS.api import TTS

if not os.path.exists('/content/fefo-repo'):
    print('[*] Baixando materiais e áudios oficiais do repositório FEFO...')
    !git clone https://github.com/elCortelini/fefo-v1.git /content/fefo-repo

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[*] Carregando o modelo neural XTTS-v2 no acelerador {device.upper()}...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('✓ Modelo XTTS-v2 carregado com sucesso!')



In [ ]:
# 3. Teste de Máxima Fidelidade com Multi-Áudios Oficiais do FEFO
import os, glob, IPython.display as ipd

dataset_dir = '/content/fefo-repo/repository/voice_fefo_dataset'

# Lista oficial de áudios para clonagem profunda (Multi-Reference Embedding)
audios_referencia = [
    os.path.join(dataset_dir, 'fefo_voz_base_minimax.wav'),
    os.path.join(dataset_dir, 'ficofelizporvc.wav'),
    os.path.join(dataset_dir, 'respeitarcolegas.wav'),
    os.path.join(dataset_dir, 'vamosfazerjuntos.wav'),
    os.path.join(dataset_dir, 'respirefundo.wav'),
]

# Filtrar apenas os que existem
audios_validos = [a for a in audios_referencia if os.path.exists(a)]
if not audios_validos:
    # Fallback
    candidatos = glob.glob('/content/fefo-repo/**/*.wav', recursive=True)
    audios_validos = [candidatos[0]] if candidatos else ['/content/fefo-repo/repository/ficofelizporvc.wav']

print(f'✓ Total de áudios de referência combinados: {len(audios_validos)} arquivos')
for i, a in enumerate(audios_validos, 1):
    print(f'   {i}. {os.path.basename(a)}')

frase_teste = 'Olá amiguinho! Eu sou o Fefo, vamos brincar juntos hoje? Estou muito feliz em te ver!'
arquivo_saida = '/content/teste_fefo_xtts.wav'

print('
[*] Gerando fala com Multi-Conditioning XTTS-v2 (Máxima Fidelidade ao timbre do FEFO)...')
tts.tts_to_file(
    text=frase_teste,
    file_path=arquivo_saida,
    speaker_wav=audios_validos,
    language='pt',
    temperature=0.45,
    repetition_penalty=2.0,
    speed=1.0
)

print('=' * 60)
print('🎉 ÁUDIO DO FEFO COM MULTI-REFERÊNCIA GERADO! CLIQUE NO PLAY ABAIXO:')
print('=' * 60)
display(ipd.Audio(arquivo_saida))



In [ ]:
# 4. Estúdio Visual WebUI do FEFO + API REST para Catalog Admin (XTTS-v2)
import os, glob, base64, gradio as gr
from fastapi import Request
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware

dataset_dir = '/content/fefo-repo/repository/voice_fefo_dataset'

# Mapeamento das opções de voz
multi_refs = [
    os.path.join(dataset_dir, 'fefo_voz_base_minimax.wav'),
    os.path.join(dataset_dir, 'ficofelizporvc.wav'),
    os.path.join(dataset_dir, 'respeitarcolegas.wav'),
    os.path.join(dataset_dir, 'vamosfazerjuntos.wav'),
    os.path.join(dataset_dir, 'respirefundo.wav'),
]
multi_refs = [a for a in multi_refs if os.path.exists(a)]

vozes_opcoes = {
    '⭐ COMBINADO: Multi-Áudios Oficiais (Recomendado - Máxima Fidelidade)': 'MULTI',
    '🏆 Base Original MiniMax (fefo_voz_base_minimax.wav)': os.path.join(dataset_dir, 'fefo_voz_base_minimax.wav'),
    '📁 Master Completo Unificado (fefo_master_combined.wav)': os.path.join(dataset_dir, 'fefo_master_combined.wav'),
    '📁 Fico Feliz por Você (ficofelizporvc.wav)': os.path.join(dataset_dir, 'ficofelizporvc.wav'),
    '📁 Respeitar Colegas (respeitarcolegas.wav)': os.path.join(dataset_dir, 'respeitarcolegas.wav'),
    '📁 Respire Fundo (respirefundo.wav)': os.path.join(dataset_dir, 'respirefundo.wav'),
    '📁 Vamos Fazer Juntos (vamosfazerjuntos.wav)': os.path.join(dataset_dir, 'vamosfazerjuntos.wav'),
    '📁 Jingle / Saudação FEFO (fefo_jingle.wav)': os.path.join(dataset_dir, 'fefo_jingle.wav'),
    '📁 Palavrinhas Mágicas (fefo_palavrinhas.wav)': os.path.join(dataset_dir, 'fefo_palavrinhas.wav'),
    '📁 Apoio e Incentivo (fefo_apoio.wav)': os.path.join(dataset_dir, 'fefo_apoio.wav'),
}

# Filtrar vozes existentes
vozes_finais = {}
for nome, caminho in vozes_opcoes.items():
    if caminho == 'MULTI' and multi_refs:
        vozes_finais[nome] = 'MULTI'
    elif os.path.exists(caminho):
        vozes_finais[nome] = caminho

if not vozes_finais:
    vozes_finais['Padrão'] = '/content/fefo-repo/repository/ficofelizporvc.wav'

def sintetizar_voz_fefo(texto, opcao_voz, velocidade, temperatura):
    if not texto or not texto.strip():
        return None
    
    caminho = vozes_finais.get(opcao_voz, 'MULTI')
    if caminho == 'MULTI' or opcao_voz == 'MULTI':
        speaker_target = multi_refs if multi_refs else '/content/fefo-repo/repository/ficofelizporvc.wav'
    else:
        speaker_target = caminho
    
    saida = '/content/fefo_webui_saida.wav'
    tts.tts_to_file(
        text=texto.strip(),
        file_path=saida,
        speaker_wav=speaker_target,
        language='pt',
        speed=float(velocidade),
        temperature=float(temperatura),
        repetition_penalty=2.0
    )
    return saida

with gr.Blocks(title='🧸 Estúdio de Voz Oficial do Mascote FEFO (XTTS-v2)') as app:
    gr.Markdown('# 🎙️ Estúdio Oficial de Voz do Mascote FEFO')
    gr.Markdown('Teste diferentes bases de voz do FEFO (inclusive a base original do MiniMax e o modo Multi-Áudio) para encontrar o timbre perfeito!')
    
    with gr.Row():
        with gr.Column(scale=1):
            txt_in = gr.Textbox(
                label='Texto para o FEFO falar:',
                value='Olá amiguinho! Eu sou o Fefo, vamos brincar juntos hoje? Estou muito feliz em te ver!',
                lines=3
            )
            voz_in = gr.Dropdown(
                label='Base de Referência da Voz:',
                choices=list(vozes_finais.keys()),
                value=list(vozes_finais.keys())[0]
            )
            with gr.Row():
                spd_in = gr.Slider(0.7, 1.4, value=1.0, step=0.05, label='Velocidade')
                temp_in = gr.Slider(0.1, 0.9, value=0.45, step=0.05, label='Expressividade (Temp)')
            
            btn = gr.Button('✨ Gerar Áudio do FEFO', variant='primary')
        
        with gr.Column(scale=1):
            audio_out = gr.Audio(label='Áudio Gerado do FEFO', type='filepath')

    btn.click(
        fn=sintetizar_voz_fefo,
        inputs=[txt_in, voz_in, spd_in, temp_in],
        outputs=[audio_out]
    )

fastapi_app = app.app
fastapi_app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

@fastapi_app.get('/health')
def api_health():
    return {'ok': True, 'status': 'online', 'engine': 'XTTS-v2', 'device': str(device), 'samples': len(multi_refs)}

@fastapi_app.post('/api/generate')
async def api_generate(req: Request):
    try:
        body = await req.json()
        texto = body.get('text', '').strip()
        if not texto:
            return JSONResponse(status_code=400, content={'ok': False, 'message': 'Texto ausente'})
        
        spd = float(body.get('speed', 1.0))
        temp = float(body.get('temperature', 0.45))
        
        # Gera o áudio com a melhor base Multi-Áudio na GPU
        caminho_wav = sintetizar_voz_fefo(texto, opcao_voz='MULTI', velocidade=spd, temperatura=temp)
        if not caminho_wav or not os.path.exists(caminho_wav):
            return JSONResponse(status_code=500, content={'ok': False, 'message': 'Erro ao gerar arquivo de áudio'})
        
        with open(caminho_wav, 'rb') as f:
            wav_bytes = f.read()
        
        b64 = base64.b64encode(wav_bytes).decode('utf-8')
        return {
            'ok': True,
            'audio_base64': b64,
            'length_bytes': len(wav_bytes),
            'engine': 'XTTS-v2-MultiRef'
        }
    except Exception as e:
        return JSONResponse(status_code=500, content={'ok': False, 'message': str(e)})

print('=' * 60)
print('🚀 INICIANDO ESTÚDIO VISUAL E API DO FEFO...')
print('💡 Copie o link público abaixo (.gradio.live) para testar no Catalog Admin!')
print('=' * 60)
app.launch(share=True, debug=False)

